# 🇪🇺 Eurostat Health Data Access

This notebook demonstrates how to access European Union health statistics from **Eurostat**, the statistical office of the European Union.

## Overview

Eurostat provides comprehensive health data covering:
- **27 EU Member States** (post-Brexit)
- **EFTA countries** (Iceland, Norway, Switzerland, Liechtenstein)
- **Candidate countries** (Turkey, Serbia, Ukraine, etc.)

### Data Available
- Healthcare expenditure and financing
- Mortality and causes of death
- Health status indicators
- Health workforce (physicians, nurses)
- Hospital beds and facilities
- Health determinants (lifestyle, environment)

**Website:** https://ec.europa.eu/eurostat/web/health

**API Docs:** https://ec.europa.eu/eurostat/web/sdmx-web-services

## 📦 Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
from pathlib import Path

# Add scripts directory to path
sys.path.insert(0, str(Path.cwd().parent.parent / 'scripts'))

from accessors import EurostatAccessor

# Initialize accessor
eurostat = EurostatAccessor()

print("✅ Eurostat Accessor initialized successfully!")

## 🌍 Exploring Eurostat Coverage

Eurostat covers EU member states plus EFTA and candidate countries:

In [ ]:
# List EU member states
eu_countries = eurostat.list_eu_countries()
print(f"🇪🇺 Total EU Member States: {len(eu_countries)}")
print("\nFirst 10 EU countries:")
eu_countries.head(10)

In [ ]:
# List all countries (EU + EFTA + Candidates)
all_countries = eurostat.list_all_countries()

print("\n🌍 Countries by Type:")
for country_type in all_countries['type'].unique():
    subset = all_countries[all_countries['type'] == country_type]
    print(f"\n{country_type} ({len(subset)} countries):")
    print(f"  {', '.join(subset['country_code'].tolist())}")

## 📊 Available Health Indicators

In [ ]:
# List common health indicators
indicators = eurostat.list_indicators()
print(f"📈 Total health indicators: {len(indicators)}")
print("\nAvailable indicators:")
indicators

In [ ]:
# Search for specific indicators
print("🔍 Searching for 'mortality' indicators:")
mortality_indicators = eurostat.search_indicators('mortality')
mortality_indicators

In [ ]:
# List causes of death
causes = eurostat.list_causes_of_death()
print("☠️  Causes of Death (ICD-10 based):")
causes

## 💰 Healthcare Expenditure

Get healthcare spending data for EU countries.

In [ ]:
# Get healthcare expenditure for selected countries
# Note: This requires internet connection and API access

try:
    expenditure = eurostat.get_healthcare_expenditure(
        countries=['DEU', 'FRA', 'ITA', 'ESP', 'NLD'],
        years=[2019, 2020, 2021, 2022]
    )
    
    if not expenditure.empty:
        print(f"Retrieved {len(expenditure)} records")
        print("\nSample data:")
        print(expenditure.head(10))
    else:
        print("No data retrieved. This may be due to API limitations.")
        
except Exception as e:
    print(f"Note: API access may be limited: {e}")
    print("The accessor is properly configured and will work when API is available.")

## 📉 Mortality Data

In [ ]:
# Get mortality data by cause
try:
    mortality = eurostat.get_mortality_data(
        cause_code='COVID-19',
        countries=['DEU', 'FRA', 'ITA', 'ESP'],
        years=[2020, 2021, 2022]
    )
    
    if not mortality.empty:
        print(f"\nRetrieved {len(mortality)} mortality records")
        print("\nSample data:")
        print(mortality.head())
    else:
        print("No mortality data retrieved from API")
        
except Exception as e:
    print(f"Note: {e}")

## 👨‍⚕️ Health Workforce

In [ ]:
# Get physician data
try:
    physicians = eurostat.get_physicians(
        countries=['DEU', 'FRA', 'ITA', 'ESP', 'POL'],
        years=[2020, 2021, 2022]
    )
    
    if not physicians.empty:
        print("Physicians Data:")
        print(physicians.head(10))
        
except Exception as e:
    print(f"Note: {e}")

In [ ]:
# Get hospital beds data
try:
    beds = eurostat.get_hospital_beds(
        countries=['DEU', 'FRA', 'ITA'],
        years=[2020, 2021]
    )
    
    if not beds.empty:
        print("Hospital Beds Data:")
        print(beds.head(10))
        
except Exception as e:
    print(f"Note: {e}")

## 📊 Life Expectancy

In [ ]:
# Get life expectancy data
try:
    life_exp = eurostat.get_life_expectancy(
        countries=['DEU', 'FRA', 'ITA', 'ESP', 'POL', 'SWE'],
        years=[2019, 2020, 2021, 2022]
    )
    
    if not life_exp.empty:
        print("Life Expectancy Data:")
        print(life_exp.head(10))
        
        # Compare across countries
        comparison = eurostat.compare_countries(
            indicator_code='demo_mlexpec',
            countries=['DEU', 'FRA', 'ITA', 'ESP', 'POL', 'SWE'],
            years=[2019, 2020, 2021, 2022]
        )
        print("\nComparison Table:")
        print(comparison)
        
except Exception as e:
    print(f"Note: {e}")

## 📈 Visualization Example

In [ ]:
# Visualize life expectancy comparison (if data available)
if not life_exp.empty and 'value' in life_exp.columns:
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for country in life_exp['geo'].unique():
        country_data = life_exp[life_exp['geo'] == country]
        ax.plot(country_data['time'], country_data['value'], 
                marker='o', label=country, linewidth=2)
    
    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('Life Expectancy (years)', fontsize=12)
    ax.set_title('Life Expectancy at Birth - Selected EU Countries', fontsize=14)
    ax.legend(title='Country', bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 🏥 Self-Perceived Health

In [ ]:
# Get self-perceived health status
try:
    self_health = eurostat.get_self_perceived_health(
        countries=['DEU', 'FRA', 'ITA', 'ESP'],
        years=[2021, 2022]
    )
    
    if not self_health.empty:
        print("Self-Perceived Health Data:")
        print(self_health.head(10))
        
except Exception as e:
    print(f"Note: {e}")

## 🔍 Advanced Usage: Generic Indicator Access

Access any Eurostat indicator by its code:

In [ ]:
# Get data dictionary for an indicator
try:
    metadata = eurostat.get_data_dictionary('demo_mlexpec')
    print("Indicator Metadata:")
    for key, value in metadata.items():
        print(f"  {key}: {value}")
        
except Exception as e:
    print(f"Note: {e}")

In [ ]:
# Get any indicator by code
try:
    # Example: Infant mortality rate
    infant_mort = eurostat.get_infant_mortality(
        countries=['DEU', 'FRA', 'ITA'],
        years=[2019, 2020, 2021]
    )
    
    if not infant_mort.empty:
        print("Infant Mortality Rate:")
        print(infant_mort.head())
        
except Exception as e:
    print(f"Note: {e}")

## 🔧 Using the eurostat Library (Optional)

For better performance, install the `eurostat` Python library:

```bash
pip install eurostat
```

The accessor will automatically use the library if available.

In [ ]:
# Check if eurostat library is available
from accessors.eurostat import EUROSTAT_LIB_AVAILABLE

if EUROSTAT_LIB_AVAILABLE:
    print("✅ eurostat library is installed and will be used")
else:
    print("⚠️  eurostat library not installed")
    print("   Install with: pip install eurostat")
    print("   Currently using REST API fallback")

## 📚 References

- **Eurostat Health:** https://ec.europa.eu/eurostat/web/health
- **Data Browser:** https://ec.europa.eu/eurostat/databrowser
- **API Documentation:** https://ec.europa.eu/eurostat/web/sdmx-web-services
- **Python Library:** https://pypi.org/project/eurostat/

## 📝 Notes

1. **Data Availability:** Some indicators may have missing data for certain countries/years.
2. **API Limits:** Eurostat API has rate limits; consider caching results for repeated use.
3. **Country Codes:** Use ISO3 codes (e.g., 'DEU' for Germany) or ISO2 (e.g., 'DE').
4. **Time Coverage:** Most indicators have annual data from 2000 onwards.